<a href="https://colab.research.google.com/github/janani26121992/AI-Projects/blob/main/AI_LSTM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Problem Statement**
Shakespeare Text Generation using LSTM with Temperature Sampling

# **Detailed Problem Statement**

The goal of this project is to build a text generation model using LSTM that learns from Shakespeare’s plays and generates new text sequences word-by-word.
The generated text should be:

1.Grammatically meaningful

2.Contextually relevant

3.Similar in style to Shakespeare

# **Necessary Libraries**
pandas, numpy → data handling

nltk → text processing

Tokenizer → convert words → numbers

LSTM → sequence learning

Embedding → word representation

EarlyStopping → prevent overtraining

In [7]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# **Data Gathering**
Dataset downloaded from Kaggle

Combined all text into one string

"I used the kagglehub library to download the Shakespeare dataset from Kaggle. Then I loaded the dataset into a pandas DataFrame, combined the text data into a single string, and previewed the first 500 characters to understand the structure of the dataset."

In [8]:
import kagglehub
path = kagglehub.dataset_download("guslovesmath/shakespeare-plays-dataset")

import os
print(os.listdir(path))

file_path = os.path.join(path, "shakespeare_plays.csv")
df = pd.read_csv(file_path)

text_data = " ".join(df['text'].dropna())
print(text_data[:500])

100%|██████████| 2.62M/2.62M [00:00<00:00, 175MB/s]

Extracting files...
['shakespeare_plays.csv']


In delivering my son from me, I bury a second husband. And I in going, madam, weep o'er my father's death anew: but I must attend his majesty's command, to whom I am now in ward, evermore in subjection. You shall find of the king a husband, madam; you, sir, a father: he that so generally is at all times good must of necessity hold his virtue to you; whose worthiness would stir it up where it wanted rather than lack it where there is such abundance. What hope is there of his majesty's amendment? 


# **Text Cleaning**

Converted text to lowercase
Removed punctuation and special characters
Keeps only alphabets → better learning

In [9]:
import re
text_data = text_data.lower()[:50000]
text_data = re.sub(r'[^a-zA-Z\s]', '', text_data)

# **Tokenization**
Converts words into numbers (tokens)

Limited vocabulary to 10,000 words (memory optimization)

<OOV> handles unknown words

In [10]:
tokenizer = Tokenizer(num_words=8000, oov_token="<OOV>")
tokenizer.fit_on_texts([text_data])

total_words = min(8000,len(tokenizer.word_index) + 1)
print("Vocabulary Size:", total_words)

# Reverse mapping: index -> word
index_word = {i: word for word, i in tokenizer.word_index.items() if i < 8000}

Vocabulary Size: 2098


# **Sequence Generation**
Converts text into sequences of numbers

Each sequence = previous words → predict next word

In [11]:
token_list = tokenizer.texts_to_sequences([text_data])[0]

input_sequences = []

for i in range(6, len(token_list)):
    n_gram_sequence = token_list[i-5:i+1]
    input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[770, 7, 126, 59, 17, 2], [7, 126, 59, 17, 2, 771], [126, 59, 17, 2, 771, 6], [59, 17, 2, 771, 6, 496], [17, 2, 771, 6, 496, 300]]


In [12]:
max_sequence_length = 7

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
Y = input_sequences[:, -1]

# Convert target to one-hot encoding
Y = to_categorical(Y, num_classes=total_words)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (9409, 5)
Y shape: (9409, 2098)


# **Model Building**
Embedding → converts words into vectors

LSTM layers → learn sequence patterns

Dense layer → final prediction

Softmax → gives probability of next word

In [13]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,)))

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=50))

# First LSTM layer
model.add(LSTM(100, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(64, dropout=0.2))

# Hidden Dense layer
model.add(Dense(64, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 6, 50)          │       104,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 6, 100)         │        60,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2098)           │       136,370 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 348,070 (1.33 MB)

 Trainable params: 348,070 (1.33 MB)

 Non-trainable params: 0 (0.00 B)

# **Model Training**
Model learns patterns from data

Early stopping prevents overfitting

Training stops automatically when no improvement

In [14]:
early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,Y,epochs=30,batch_size=32,verbose=1,callbacks=[early_stop])

Epoch 1/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.0278 - loss: 6.6671
Epoch 2/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0264 - loss: 6.2745
Epoch 3/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0266 - loss: 6.2019
Epoch 4/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0276 - loss: 6.0946
Epoch 5/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0297 - loss: 6.0041
Epoch 6/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0392 - loss: 5.9039
Epoch 7/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0419 - loss: 5.7693
Epoch 8/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0433 - loss: 5.6486
Epoch 9/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0440 - loss: 5.5439
Epoch 10/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0483 - loss: 5.4481
Epoch 11/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0488 - loss: 5.3579
Epoch 12/30
295/295 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/s

In [15]:
model.save("TextGenerationModel1.keras")

# **Temperature & Top-K Sampling**

1.Temperature is a hyperparameter that controls randomness in text generation.
Temperature controls randomness

Low → predictable text

High → creative text


2.Top-K sampling selects the K most probable words and randomly chooses from them, improving quality while maintaining diversity.

In [16]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")


    preds = np.log(preds + 1e-10) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)


    top_indices = np.argsort(preds)[-top_k:]
    top_probs = preds[top_indices]


    top_probs = top_probs / np.sum(top_probs)

    return np.random.choice(top_indices, p=top_probs)

# **Text Generation**

In [17]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)[0]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.8,
            top_k=5
        )

        next_word = index_word.get(predicted_index, "")

        # avoid immediate repetition
        if next_word in generated_words[-3:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

In [18]:
print(generate_text("my lord", next_words=10))

my lord that the majesty will god so soon i am you


In [19]:
print(generate_text("the king shall", next_words=20))

the king shall have so one you have him i know me you all the very lord that is bear


# ***“I will check What happens when temperature changes?***

In [20]:
def generate_text2(seed_text, next_words=20, temperature=0.8, top_k=5):
    output_text = seed_text

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)[0]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=temperature,
            top_k=top_k
        )

        next_word = index_word.get(predicted_index, "")

        if next_word == "":
            continue

        output_text += " " + next_word

    return output_text

In [21]:
print("Temperature = 0.5")
print(generate_text2("my lord", 10, temperature=0.5))

print("\nTemperature = 0.8")
print(generate_text2("my lord", 10, temperature=0.8))

print("\nTemperature = 1.2")
print(generate_text2("my lord", 10, temperature=1.2))

Temperature = 0.5
my lord lustig the lottery is my lord i wilt not commend

Temperature = 0.8
my lord greets is formerly fire and caution me it the sinister

Temperature = 1.2
my lord if what so beg you the life in i have


Low temperature (e.g., 0.5):

→ Model selects high-probability words
→ Output is predictable and repetitive

Medium temperature (e.g., 0.8):

→ Balanced selection
→ Output is natural and meaningful

High temperature (e.g., 1.2):

→ More randomness
→ Output is creative but may be grammatically incorrect

In [23]:
def generate_text_greedy(seed_text, next_words=20):
    output_text = seed_text

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)[0]


        predicted_index = np.argmax(predicted_probs)

        next_word = index_word.get(predicted_index, "")

        output_text += " " + next_word

    return output_text

In [24]:
print("Without Temperature:")
print(generate_text_greedy("my lord", 10))

print("\nWith Temperature:")
print(generate_text("my lord", 10))

Without Temperature:
my lord lustig the confirmation of the king to would not the

With Temperature:
my lord shall replete i am not before you know i


Model performance without temperature can be evaluated using greedy sampling (argmax), where the word with the highest probability is selected at each step. This produces deterministic but less diverse output compared to temperature-based sampling.